# S09 — External-CHM GEDI-support sensitivity

Evaluates how the matched GEDI support affects the external-product benchmark while keeping product comparisons paired within each landscape.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import hashlib, json, math, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform as crs_transform
from rasterio.windows import Window
from shapely.geometry import Point, box
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
ROOT = PROJECT / "Ablations" / "PreSubmission_Independent_Robustness"
OUT = ROOT / "results" / "external_chm_support_sensitivity"
TABLES, FIGURES = OUT/"tables", OUT/"figures"
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = PROJECT/"Results/Final_Article_Harmonized_GEDIAnchored_NaturalP1/CHM_Comparison/GEDI_TEST_product_valid_support_signed_errors.csv.gz"
RUN_EXTRACTION = True
MIN_FOOTPRINT_COVERAGE = 0.80
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260818

PRODUCT_ROOT=PROJECT/"CHM_Products_Comparison"
SITES={
 "Ifran":{"key":"Ifran_6","products":["Our Model","Pa24","L23","T24","P21"]},
 "Maamoura":{"key":"Maamoura","products":["Our Model","Pa24","L23","T24","P21"]},
 "Agadir":{"key":"Agadir","products":["Our Model","Pa24","L23","T24"]}}
for forest,cfg in SITES.items():
 tag="EPSG32630" if forest=="Ifran" else "EPSG32629"
 root=PRODUCT_ROOT/cfg["key"]
 cfg["maps"]={
  "L23":root/f"ETH_Lang_2020_CHM_10m/clean/mosaic/{cfg['key']}__ETH_Lang_2020_CHM_10m__clean__EPSG32630.tif",
  "T24":root/f"Meta_WRI_Tolan_2023_CHM_resampled_10m/clean/mosaic/{cfg['key']}__Meta_WRI_Tolan_2023_CHM_resampled_10m__clean__EPSG32630.tif",
  "P21":root/f"GFCH_Potapov_GLAD_2019_30m/clean/mosaic/{cfg['key']}__GFCH_Potapov_GLAD_2019_30m__clean__EPSG32630.tif",
  "Pa24":PRODUCT_ROOT/f"_PAULS_2020_VERIFIED_V1/{forest}/Pauls_et_al_2024_CHM_2020_10m/clean/mosaic/{forest}__Pauls_et_al_2024_CHM_2020_10m__clean__{tag}.tif"}

base=pd.read_csv(CACHE)
points=(base.sort_values(["forest","shot_id"]).drop_duplicates(["forest","shot_id"])
        [["forest","shot_id","rh95","gedi_year","lon","lat"]].copy())
print("Canonical points:", points.groupby("forest").size().to_dict())

In [ ]:
def valid_value(a, ds):
    a=np.asarray(a,dtype=float)
    ok=np.isfinite(a)
    if ds.nodata is not None and np.isfinite(ds.nodata): ok &= a != ds.nodata
    ok &= a != -9999
    return a,ok

def sample_one_raster(path, frame, radius=12.5):
    rows=[]
    with rasterio.open(path) as ds:
        xs,ys=crs_transform("EPSG:4326",ds.crs,frame.lon.tolist(),frame.lat.tolist())
        for rec,x,y in zip(frame.itertuples(index=False),xs,ys):
            rr,cc=ds.index(x,y)
            nearest=np.nan; bilinear=np.nan; circle=np.nan; coverage=0.0
            if 0<=rr<ds.height and 0<=cc<ds.width:
                a,ok=valid_value(ds.read(1,window=Window(cc,rr,1,1)),ds)
                if ok.any(): nearest=float(a[ok][0])
                # Four closest pixel centres, with inverse bilinear weights.
                inv=~ds.transform; cf,rf=inv*(x,y); c0=math.floor(cf-.5); r0=math.floor(rf-.5)
                vals=[]; weights=[]
                for r in (r0,r0+1):
                    for c in (c0,c0+1):
                        if 0<=r<ds.height and 0<=c<ds.width:
                            v,m=valid_value(ds.read(1,window=Window(c,r,1,1)),ds)
                            xc,yc=ds.xy(r,c); w=max(0,1-abs(cf-(c+.5)))*max(0,1-abs(rf-(r+.5)))
                            if m.any() and w>0: vals.append(float(v[m][0])); weights.append(w)
                if weights: bilinear=float(np.average(vals,weights=weights))
                footprint=Point(x,y).buffer(radius,resolution=32); pix_area=[]; pix_val=[]; valid_area=0.0
                rb=int(math.ceil(radius/abs(ds.transform.e)))+2; cb=int(math.ceil(radius/abs(ds.transform.a)))+2
                for r in range(max(0,rr-rb),min(ds.height,rr+rb+1)):
                    for c in range(max(0,cc-cb),min(ds.width,cc+cb+1)):
                        x0,y0=ds.xy(r,c,offset="ul"); x1,y1=ds.xy(r,c,offset="lr")
                        area=footprint.intersection(box(min(x0,x1),min(y0,y1),max(x0,x1),max(y0,y1))).area
                        if area<=0: continue
                        v,m=valid_value(ds.read(1,window=Window(c,r,1,1)),ds)
                        if m.any(): pix_area.append(area); pix_val.append(float(v[m][0])); valid_area+=area
                coverage=valid_area/footprint.area
                if pix_area and coverage>=MIN_FOOTPRINT_COVERAGE: circle=float(np.average(pix_val,weights=pix_area))
            rows.append((str(rec.shot_id),nearest,bilinear,circle,coverage))
    return pd.DataFrame(rows,columns=["shot_id","native_nearest","native_bilinear","circle12p5_area_weighted","circle_coverage"])

def our_annual_path(forest,year):
    eco={"Ifran":"Dense","Maamoura":"Low_Sparsity","Agadir":"Sparse"}[forest]
    names={
     "Ifran":f"Ifran_B4_C15_Phase2_Y{year}_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
     "Maamoura":f"Maamoura_B4_C15_Phase2_Y{year}_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
     "Agadir":f"Agadir_B4_C15_Phase2_Y{year}_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif"}
    return PROJECT/f"Inference_Harmonized_GEDIAnchored_NaturalP1/{eco}/{forest}/Phase2/Y{year}/Annual"/names[forest]

long_path=TABLES/"01_all_products_all_supports.csv.gz"
if RUN_EXTRACTION:
    blocks=[]
    for forest,cfg in SITES.items():
        fp=points[points.forest.eq(forest)].copy(); fp["shot_id"]=fp.shot_id.astype(str)
        for product in cfg["products"]:
            pieces=[]
            if product=="Our Model":
                for year,yp in fp.groupby("gedi_year"):
                    path=our_annual_path(forest,int(year));
                    if not path.is_file(): raise FileNotFoundError(path)
                    s=sample_one_raster(path,yp); pieces.append(s)
            else:
                path=cfg["maps"][product]
                if not path.is_file(): raise FileNotFoundError(path)
                pieces=[sample_one_raster(path,fp)]
            s=pd.concat(pieces,ignore_index=True); s["forest"]=forest; s["product"]=product
            blocks.append(fp[["shot_id","rh95"]].merge(s,on="shot_id",how="left"))
            print(forest,product,"done")
    wide=pd.concat(blocks,ignore_index=True)
    long=wide.melt(id_vars=["forest","product","shot_id","rh95","circle_coverage"],
                   value_vars=["native_nearest","native_bilinear","circle12p5_area_weighted"],
                   var_name="support_method",value_name="prediction")
    long.to_csv(long_path,index=False,compression="gzip")
else:
    long=pd.read_csv(long_path,dtype={"shot_id":str})
display(long.groupby(["forest","product","support_method"]).prediction.count().unstack())

In [ ]:
def regression_metrics(frame, pred_col="prediction"):
    y = pd.to_numeric(frame["rh95"], errors="coerce").to_numpy(float)
    p = pd.to_numeric(frame[pred_col], errors="coerce").to_numpy(float)
    ok = np.isfinite(y) & np.isfinite(p); y, p = y[ok], p[ok]
    if len(y) < 3: return {"n":len(y), **{k:np.nan for k in ("mae","rmse","bias","r2","r","slope","std_ratio")}}
    e=p-y; sy=np.std(y)
    return {"n":len(y), "mae":np.mean(np.abs(e)), "rmse":np.sqrt(np.mean(e*e)),
            "bias":np.mean(e), "r2":1-np.sum(e*e)/np.sum((y-y.mean())**2),
            "r":np.corrcoef(y,p)[0,1], "slope":np.polyfit(y,p,1)[0],
            "std_ratio":np.std(p)/sy if sy>0 else np.nan}

In [ ]:
strict=[]
for forest,cfg in SITES.items():
    sub=long[long.forest.eq(forest)].copy(); expected={(p,m) for p in cfg["products"] for m in sub.support_method.unique()}
    support_sets={(p,m):set(g.shot_id[g.prediction.notna()].astype(str)) for (p,m),g in sub.groupby(["product","support_method"])}
    missing=expected-set(support_sets)
    if missing: raise RuntimeError(f"{forest}: missing combinations {missing}")
    common=set.intersection(*(support_sets[k] for k in sorted(expected)))
    if len(common)<30: raise RuntimeError(f"{forest}: strict all-product/all-operator support n={len(common)}")
    strict.append(sub[sub.shot_id.astype(str).isin(common)].copy())
strict=pd.concat(strict,ignore_index=True)
strict.to_csv(TABLES/"02_strict_common_all_products_all_supports.csv.gz",index=False,compression="gzip")

rows=[]
for keys,g in strict.groupby(["forest","support_method","product"]): rows.append({"forest":keys[0],"support_method":keys[1],"product":keys[2],**regression_metrics(g)})
metrics=pd.DataFrame(rows)
metrics["rank_mae"]=metrics.groupby(["forest","support_method"]).mae.rank(method="min")
metrics["rank_rmse"]=metrics.groupby(["forest","support_method"]).rmse.rank(method="min")
metrics.to_csv(TABLES/"03_metrics_and_ranks_strict_common.csv",index=False)
display(metrics.sort_values(["forest","support_method","rank_mae"]))

rng=np.random.default_rng(BOOTSTRAP_SEED); boot=[]
for forest,gf in strict.groupby("forest"):
    shots=np.asarray(sorted(gf.shot_id.astype(str).unique()))
    for method,gm in gf.groupby("support_method"):
        pivot=gm.pivot(index="shot_id",columns="product",values="prediction").loc[shots]
        truth=gm.drop_duplicates("shot_id").set_index("shot_id").loc[shots,"rh95"].to_numpy(float)
        wins={p:0 for p in pivot.columns}
        for _ in range(BOOTSTRAP_REPLICATES):
            ix=rng.integers(0,len(shots),len(shots)); mae={p:np.mean(np.abs(pivot[p].to_numpy()[ix]-truth[ix])) for p in pivot}
            wins[min(mae,key=mae.get)]+=1
        for p,nwin in wins.items(): boot.append({"forest":forest,"support_method":method,"product":p,"p_rank1_mae":nwin/BOOTSTRAP_REPLICATES})
boot=pd.DataFrame(boot); boot.to_csv(TABLES/"04_bootstrap_probability_rank1_mae.csv",index=False)

fig,axes=plt.subplots(1,3,figsize=(15,4.8),sharey=False)
for ax,(forest,g) in zip(axes,metrics.groupby("forest",sort=False)):
    for product,gp in g.groupby("product"):
        gp=gp.set_index("support_method").reindex(["native_nearest","native_bilinear","circle12p5_area_weighted"])
        ax.plot(["Native\nnearest","Native\nbilinear","GEDI disk\n12.5 m"],gp.rank_mae,marker="o",label=product)
    ax.invert_yaxis(); ax.set_title(forest); ax.set_ylabel("MAE rank (1 = best)"); ax.grid(alpha=.2)
axes[-1].legend(bbox_to_anchor=(1.02,1),loc="upper left"); fig.suptitle("Sensitivity of product ranking to GEDI extraction support")
fig.tight_layout(); fig.savefig(FIGURES/"product_rank_sensitivity.pdf",bbox_inches="tight"); fig.savefig(FIGURES/"product_rank_sensitivity.png",dpi=300,bbox_inches="tight")
plt.show()

decision=(metrics.groupby(["forest","product"]).agg(rank_min=("rank_mae","min"),rank_max=("rank_mae","max"),mae_min=("mae","min"),mae_max=("mae","max")).reset_index())
decision["rank_changes"]=decision.rank_min.ne(decision.rank_max)
decision.to_csv(TABLES/"05_ranking_sensitivity_decision.csv",index=False); display(decision)